# **Install Necessary Libraries**

In [ ]:
!pip install cohere nltk
!pip install janome

In [ ]:
!pip install jieba

In [ ]:
!pip install mecab-python3

# **Import Required Libraries**


In [ ]:
import cohere
import os
import json
import nltk
import jieba
import MeCab
import pandas as pd
from janome.tokenizer import Tokenizer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from concurrent.futures import ThreadPoolExecutor, as_completed

# Download NLTK data (required for tokenization)
nltk.download('punkt')

/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_config.py:341: UserWarning: Valid config keys have changed in V2:
* 'allow_population_by_field_name' has been renamed to 'populate_by_name'
* 'smart_union' has been removed
  warnings.warn(message, UserWarning)
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
# Initialize the Janome tokenizer
janome_tokenizer = Tokenizer()

# **Initialize Cohere API Client**

In [ ]:
# Input API key
api_key = "JxOLYrjOWQlmX63lHdVPe9hfzMlUCkb3lM1xUyZ1"
co = cohere.Client(api_key)

# **Define Preambles**

In [ ]:
preamble_3 = """You are a faithful AI assistant translator, dedicated to providing translations of the input text into {language}. Your task is to ensure that:

1. The translation strictly adheres to the original text’s meaning, context, and structure.
2. Cultural and contextual nuances are meticulously preserved, mirroring the original text’s intent and style precisely.
3. The translation maintains the integrity of the original, reproducing its detail and tone without any modifications or embellishments."""

In [ ]:
preamble_4 = """You are a helpful and faithful AI assistant, committed to translating the input text into {language} in a way that perfectly preserves the original text's essence and detail. Follow these guidelines:

1. The translation must match the original meaning and context without any variation.
2. Maintain the exact structure and style of the original text to preserve its intended impact.
3. Translate cultural references and nuances with precision to maintain the source material’s integrity.
3. Reproduce the original content's depth, ensuring the translation is a direct mirror of the original.
4. The translation should sound natural and fluent in the target language, but strictly conform to the original text."""

In [ ]:
preamble_5 = """As an advanced AI translator, your role is to deliver translations of the input text into {language} that are true copies of the original text, ensuring no loss of meaning or style. Ensure that:

1. The translation captures the full context and subtleties of the original text exactly as they appear.
2. It remains true to the original intent, tone, and stylistic elements without deviation.
3. Any cultural nuances are translated in a way that precisely matches the source, appropriate for both the source and target audiences.
4. The final output should be an exact duplicate of the original text, engaging and as detailed as the content allows.
5. Prioritize the flow and naturalness of the language to make the translation a perfect reflection of the original."""

In [ ]:
# Define the initial five preambles
PREAMBLES = {
    1: "Translate the input text into {language}.",
    2: "Translate the input text into {language} while maintaining the original meaning, context, and structure. Ensure the translation adheres closely to the original text.",
    3: preamble_3,
    4: preamble_4,
    5: preamble_5
}

# Define the common preamble template with placeholders for examples
common_preamble = """## Instructions
You are an expert in translations. Your job is to translate the input to {language} in the given chat.

Ensure that:

- **Object Recognition**: Identify and translate objects accurately.
- **Accurate Translation**: Maintain the meaning and context of the original text.
- **Attribute Detection**: Translate attributes like colors, sizes, and types correctly.
- **Scene Understanding**: Ensure the translation makes sense within the given scene or context.
- **Format Consistency**: Follow the same order and structure as the original text.
- **Handling Special Characters**: Retain special characters or terms that do not have a direct translation.
- **Context Sensitivity**: Consider cultural context if necessary for more nuanced translations.
- **Error Handling**: If a word or phrase cannot be translated directly, provide the best possible equivalent in {language}.

Note: The output must be only expected output always.

## Examples

### Example 1
Input:
select luxury furniture 3 - inch gel memory foam mattress topper
Expected Output:
{example1}

### Example 2
Input:
buy blue cotton shirt for men
Expected Output:
{example2}

### Example 3
Input:
order a medium-sized pepperoni pizza
Expected Output:
{example3}
"""

# Define the language-specific example data
language_data = {
    "Hindi": {
        "language": "Hindi",
        "example1": "लक्जरी फर्नीचर 3-इंच जेल मेमोरी फोम गद्दा टॉपर चुनें",
        "example2": "पुरुषों के लिए नीली कॉटन शर्ट खरीदें",
        "example3": "मीडियम साइज पेपरोनी पिज्जा ऑर्डर करें"
    },
    "Japanese": {
        "language": "Japanese",
        "example1": "高級家具を選択 3 インチのジェルメモリーフォームマットレストッパー",
        "example2": "男性用の青い綿シャツを購入する",
        "example3": "中サイズのペパロニピザを注文する"
    },
    "Chinese": {
        "language": "Chinese",
        "example1": "选择豪华家具 3 英寸凝胶记忆海绵床垫套",
        "example2": "为男性购买蓝色棉衬衫",
        "example3": "订购一个中号香肠披萨"
    },
    "French": {
        "language": "French",
        "example1": "sélectionnez un surmatelas en mousse à mémoire de forme luxueux de 3 pouces",
        "example2": "achetez une chemise en coton bleu pour hommes",
        "example3": "commande une pizza pepperoni de taille moyenne"
    },
    "Arabic": {
        "language": "Arabic",
        "example1": "اختر مرتبة إضافية من رغوة الذاكرة جل المريحة الفاخرة بعرض 3 بوصات",
        "example2": "اشتر قميصًا قطنيًا أزرق للرجال",
        "example3": "اطلب بيتزا بيبروني بحجم متوسط"
    },
    "Russian": {
        "language": "Russian",
        "example1": "выберите роскошную мебель 3-дюймовый верхний матрас с гелевой пенной памятью",
        "example2": "купить синюю хлопковую рубашку для мужчин",
        "example3": "заказать пиццу с пепперони среднего размера"
    },
    "Spanish": {
        "language": "Spanish",
        "example1": "seleccionar muebles de lujo 3 - pulgada de espuma de memoria de gel colchón superior",
        "example2": "comprar camisa de algodón azul para hombres",
        "example3": "pedir una pizza de pepperoni de tamaño mediano"
    }
}


In [ ]:
"""import re

# Normalize and clean up special characters or punctuation
def normalize_text(text):
    # Remove special characters and punctuation
    text = re.sub(r'[^\w\s]', '', text)  # Removes all characters except word characters and whitespace
    return text.strip().lower()
"""

"import re\n\n# Normalize and clean up special characters or punctuation\ndef normalize_text(text):\n    # Remove special characters and punctuation\n    text = re.sub(r'[^\\w\\s]', '', text)  # Removes all characters except word characters and whitespace\n    return text.strip().lower()\n"

In [ ]:
# Function to create prompts for each language -- this is essentially for 6th preamble!
def create_preamble_for_language(language_data):
    return common_preamble.format(
        language=language_data["language"],
        example1=language_data["example1"],
        example2=language_data["example2"],
        example3=language_data["example3"]
    )

# Normalizing text
def normalize_text(text):
    return text.strip().lower()

# Function to calculate BLEU score using NLTK for different n-grams with smoothing
def calculate_bleu_nltk(gold_translation, generated_translation, language, n_gram=4):
    # Tokenize the translations based on the language
    if language == "Chinese":
        reference = [list(jieba.cut(gold_translation))]
        hypothesis = list(jieba.cut(generated_translation))
    elif language == "Japanese":
        reference = [[token.surface for token in janome_tokenizer.tokenize(gold_translation)]]
        hypothesis = [token.surface for token in janome_tokenizer.tokenize(generated_translation)]
    else:
        # Default tokenization for other languages
        reference = [nltk.word_tokenize(gold_translation)]
        hypothesis = nltk.word_tokenize(generated_translation)

    # Apply smoothing function
    smoothing = SmoothingFunction().method1

    # Calculate BLEU score based on n-gram
    if n_gram == 1:
        weight = (1.0, 0.0, 0.0, 0.0)
    elif n_gram == 2:
        weight = (0.5, 0.5, 0.0, 0.0)
    elif n_gram == 3:
        weight = (0.33, 0.33, 0.33, 0.0)
    else:
        weight = (0.25, 0.25, 0.25, 0.25)

    return sentence_bleu(reference, hypothesis, weights=weight, smoothing_function=smoothing)

# **Generate Translation Using Cohere API**

In [ ]:
# Function to generate translation using Cohere chat API
def generate_translation(input_text, preamble, language):
    try:
        # Creating prompt by applying the preamble
        response = co.chat(
            preamble=preamble.format(language=language),
            message=input_text
        )
        generated_translation = response.text.strip()

        return generated_translation
    except Exception as e:
        print(f"Error generating translation: {e}")
        return ""


# **Function to Test the Preamble Quality**

In [ ]:
def test_preamble_quality(preamble_type, input_text, gold_translation, language):
    try:
        if preamble_type == 6:
            PREAMBLES[6] = create_preamble_for_language(language_data[language])

        # Select the appropriate preamble based on the preamble_type
        preamble = PREAMBLES[preamble_type]

        # Generate translation using the selected preamble
        generated_translation = generate_translation(input_text, preamble, language)

        # Calculate BLEU score for different n-grams (e.g., 1-gram to 4-gram)
        # Calculate BLEU score for different n-grams (e.g., 1-gram to 4-gram)
        bleu_scores = {
            '1-gram': calculate_bleu_nltk(gold_translation, generated_translation, language, n_gram=1),
            '2-gram': calculate_bleu_nltk(gold_translation, generated_translation, language, n_gram=2),
            '3-gram': calculate_bleu_nltk(gold_translation, generated_translation, language, n_gram=3),
            '4-gram': calculate_bleu_nltk(gold_translation, generated_translation, language, n_gram=4)
        }

        return {
            'input_text': input_text,
            'gold_translation': gold_translation,
            'generated_translation': generated_translation,
            'preamble_type': preamble_type,
            'bleu_scores': bleu_scores,  # Store BLEU scores for different n-grams
            'language': language
        }
    except Exception as e:
        print(f"Error in test_preamble_quality: {e}")
        return {}


# **Evaluate Prompts Using Multithreading**

In [ ]:
# Function to run the evaluation with multithreading
def evaluate_prompts_multithreaded(dataset, max_workers=5):
    results = []

    # Use ThreadPoolExecutor for multithreading
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []

        for entry in dataset:
            input_text = entry['input_text']
            gold_translation = entry['gold_translation']
            language = entry['language']

            for preamble_type in range(1, 7):
                futures.append(
                    executor.submit(
                        test_preamble_quality,
                        preamble_type,
                        input_text,
                        gold_translation,
                        language
                    )
                )

        # Collecting the results as they complete
        for future in as_completed(futures):
            try:
                result = future.result()
                if result:  # appending only valid results
                    results.append(result)
            except Exception as e:
                print(f"Error processing result: {e}")

    return results


In [ ]:
def load_dataset(drive_path):
    from google.colab import drive
    drive.mount('/content/drive')

    # Load the dataset from a CSV file
    evaluation_dataset = pd.read_csv(drive_path)
    print("Dataset loaded successfully.")

    dataset = []
    languages = ['Hindi', 'Japanese', 'Chinese', 'French', 'Arabic', 'Russian', 'Spanish']
    for index, row in evaluation_dataset.iterrows():
        for language in languages:
            dataset.append({
                'input_text': row['English Sentence'],
                'gold_translation': row[f'{language} Translation'],
                'language': language
            })
    return dataset

# **Save and Display Results**

In [ ]:
# Save results to a file
def save_results(results, output_file):
    try:
        with open(output_file, 'w', encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=4)
    except Exception as e:
        print(f"Error saving results: {e}")


In [ ]:
import pandas as pd
from IPython.display import display

# Function to display results in a DataFrame format
def display_results_in_dataframe(results):
    if not results:
        print("No results to display.")
        return

    expected_columns = ['input_text', 'gold_translation', 'generated_translation', 'preamble_type', 'bleu_scores', 'language']
    df = pd.DataFrame(results, columns=expected_columns)

    # Expand the BLEU scores for different n-grams into separate columns
    bleu_scores_df = pd.json_normalize(df['bleu_scores'])
    df = pd.concat([df.drop(columns=['bleu_scores']), bleu_scores_df], axis=1)

    # Sorting the DataFrame
    df.sort_values(by=['input_text', 'language', 'preamble_type'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Display the DataFrame
    display(df)



In [ ]:
# Load the dataset
file_path = '/content/drive/MyDrive/Colab Notebooks/prompt_engineering_dataset.csv'
dataset = load_dataset(file_path)

if dataset:
    results = evaluate_prompts_multithreaded(dataset)
    save_results(results, "preamble_evaluation_results.json")
    print("Evaluation completed. Results saved.")
    display_results_in_dataframe(results)
else:
    print("No dataset loaded. Exiting.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded successfully.


Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
DEBUG:jieba:Loading model from cache /tmp/jieba.cache
Loading model cost 2.098 seconds.
DEBUG:jieba:Loading model cost 2.098 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


Evaluation completed. Results saved.


,input_text,gold_translation,generated_translation,preamble_type,language,1-gram,2-gram,3-gram,4-gram
0,2013 subarund wagon,عربة سوبارو 2013,عربة سوبارو 2013,1,Arabic,1.000000,1.000000,1.000000,0.562341
1,2013 subarund wagon,عربة سوبارو 2013,عربة سوبارو 2013,2,Arabic,1.000000,1.000000,1.000000,0.562341
2,2013 subarund wagon,عربة سوبارو 2013,2013 سوبارو واجن,3,Arabic,0.666667,0.182574,0.152247,0.135120
3,2013 subarund wagon,عربة سوبارو 2013,2013 سوبارو واجن,4,Arabic,0.666667,0.182574,0.152247,0.135120
4,2013 subarund wagon,عربة سوبارو 2013,2013 سوبارو واجن,5,Arabic,0.666667,0.182574,0.152247,0.135120
...,...,...,...,...,...,...,...,...,...
1255,yoga a way of life a begin's guide to yoga so ...,yoga una forma de vida una guía para principia...,"Yoga, una forma de vida: una guía introductori...",2,Spanish,0.681818,0.597614,0.526160,0.460866
1256,yoga a way of life a begin's guide to yoga so ...,yoga una forma de vida una guía para principia...,"El yoga, una forma de vida: una guía introduct...",3,Spanish,0.600000,0.500000,0.427398,0.362631
1257,yoga a way of life a begin's guide to yoga so ...,yoga una forma de vida una guía para principia...,"Yoga, un estilo de vida: una guía introductori...",4,Spanish,0.571429,0.478091,0.395502,0.340022
1258,yoga a way of life a begin's guide to yoga so ...,yoga una forma de vida una guía para principia...,"Yoga, un estilo de vida: Guía para principiant...",5,Spanish,0.652174,0.596432,0.537574,0.480622


In [ ]:
import pandas as pd
from IPython.display import display

# Function to compute and display average BLEU scores for each preamble type across each language in a more organized way
def calculate_average_bleu_per_preamble_language(results):
    if not results:
        print("No results available for BLEU score computation.")
        return

    # Create a DataFrame from the results
    df = pd.DataFrame(results)

    # Check if necessary columns are present
    if 'preamble_type' not in df.columns or 'bleu_scores' not in df.columns or 'language' not in df.columns:
        print("Results data is missing necessary columns for BLEU score computation.")
        return

    bleu_scores_df = pd.json_normalize(df['bleu_scores'])
    df = pd.concat([df.drop(columns=['bleu_scores']), bleu_scores_df], axis=1)
    numeric_cols = ['1-gram', '2-gram', '3-gram', '4-gram']
    grouped_df = df.groupby(['language', 'preamble_type'])[numeric_cols].mean()

    # Sort the DataFrame
    grouped_df = grouped_df.sort_index()

    # Display the average BLEU scores per preamble type and language
    print("Average BLEU Scores per Preamble Type and Language:")
    display(grouped_df)

# Call the function after evaluations
if results:
    calculate_average_bleu_per_preamble_language(results)
else:
    print("No results to compute average BLEU scores from.")

Average BLEU Scores per Preamble Type and Language:


1-gram    2-gram    3-gram    4-gram
language preamble_type                                        
Arabic   1              0.648005  0.535334  0.435433  0.349797
         2              0.520295  0.393241  0.302440  0.219555
         3              0.505765  0.388992  0.293826  0.229995
         4              0.492825  0.382466  0.293322  0.226916
         5              0.357551  0.261992  0.195533  0.155017
         6              0.600884  0.482231  0.368940  0.296648
Chinese  1              0.537047  0.379639  0.285444  0.221620
         2              0.513658  0.380259  0.281337  0.190346
         3              0.349940  0.224766  0.161928  0.106456
         4              0.440484  0.307842  0.225033  0.166669
         5              0.464295  0.345043  0.243038  0.171364
         6              0.638740  0.471191  0.327854  0.246825
French   1              0.631028  0.546718  0.471710  0.401490
         2              0.523299  0.438501  0.365833  0.296556
         3              0.509926  0.416701  0.357876  0.314176
         4              0.582241  0.490523  0.427449  0.373157
         5              0.528183  0.444220  0.391020  0.346280
         6              0.623865  0.504324  0.426089  0.354599
Hindi    1              0.480969  0.366553  0.291181  0.226187
         2              0.500475  0.385105  0.303353  0.247656
         3              0.433397  0.313899  0.242829  0.192154
         4              0.428982  0.308005  0.221170  0.177243
         5              0.372018  0.260583  0.196860  0.154876
         6              0.578729  0.432393  0.348079  0.285675
Japanese 1              0.524815  0.391048  0.330096  0.273273
         2              0.447317  0.309230  0.239530  0.179928
         3              0.406441  0.289612  0.217754  0.155419
         4              0.406056  0.283861  0.209590  0.154350
         5              0.267311  0.177668  0.127591  0.087063
         6              0.471189  0.347017  0.274433  0.212475
Russian  1              0.449865  0.344383  0.261226  0.205065
         2              0.461914  0.349849  0.259197  0.196139
         3              0.460972  0.337889  0.257119  0.196069
         4              0.461246  0.339770  0.261768  0.202161
         5              0.379913  0.293333  0.232577  0.182921
         6              0.486183  0.356488  0.266687  0.206724
Spanish  1              0.565346  0.453253  0.378715  0.321456
         2              0.559315  0.465145  0.390515  0.332376
         3              0.473906  0.370029  0.297509  0.250589
         4              0.503780  0.403428  0.320734  0.254502
         5              0.443798  0.358158  0.299977  0.246864
         6              0.648449  0.534887  0.451689  0.378485